In [3]:
import sys

sys.path.append('d:\\Pl_Capitals_Assignment\\portfolio-analytics-agent')

In [4]:
from src.database.db_connection import get_db_connection

cur = get_db_connection().cursor()

cur.execute("""
SELECT portfolio_name FROM portfolios""")

rows = cur.fetchall()

rows

2026-04-05 19:04:55,438 | INFO | src.database.db_connection | Connected to database: D:\Pl_Capitals_Assignment\portfolio-analytics-agent\data\db\portfolio_database.db


[('Balanced Portfolio',),
 ('Conservative Income Fund',),
 ('Dividend Aristocrats Fund',),
 ('ESG Sustainable Fund',),
 ('Emerging Markets Fund',),
 ('Fixed Income Plus',),
 ('Growth Equity Fund',),
 ('International Equity Fund',),
 ('Small Cap Value Fund',),
 ('Tech Innovation Fund',),
 ('Total Bond Market Index Fund',),
 ('Total International Index Fund',),
 ('Total Stock Market Index Fund',)]

In [6]:
cur.execute("SELECT COUNT(*) FROM portfolios")
cur.fetchall()

[(13,)]

In [9]:
cur.execute("SELECT p.portfolio_id, p.portfolio_name, COUNT(DISTINCT s.sector_id) AS num_sectors, COUNT(h.security_id) AS total_holdings, COUNT(DISTINCT s.sector_id) / COUNT(h.security_id) AS diversification_ratio FROM portfolios p JOIN holdings h ON p.portfolio_id = h.portfolio_id JOIN securities s ON h.security_id = s.security_id WHERE s.asset_type = 'Stock' GROUP BY p.portfolio_id, p.portfolio_name HAVING COUNT(DISTINCT s.sector_id) > 5")
cur.fetchall()

[(11, 'Total Stock Market Index Fund', 6, 12, 0)]

In [4]:
cur.execute("WITH portfolio_sector_count AS (SELECT p.portfolio_id, p.portfolio_name, COUNT(DISTINCT sec.sector_id) as sector_count FROM portfolios p JOIN holdings h ON p.portfolio_id = h.portfolio_id JOIN securities s ON h.security_id = s.security_id JOIN sectors sec ON s.sector_id = sec.sector_id WHERE s.asset_type = 'Stock' GROUP BY p.portfolio_id, p.portfolio_name HAVING COUNT(DISTINCT sec.sector_id) > 5), portfolio_holdings_count AS (SELECT p.portfolio_id, COUNT(h.holding_id) as total_holdings FROM portfolios p JOIN holdings h ON p.portfolio_id = h.portfolio_id GROUP BY p.portfolio_id) SELECT psc.portfolio_name, psc.sector_count, phc.total_holdings, ROUND(CAST(psc.sector_count AS REAL) / phc.total_holdings, 3) as diversification_ratio FROM portfolio_sector_count psc JOIN portfolio_holdings_count phc ON psc.portfolio_id = phc.portfolio_id ORDER BY diversification_ratio DESC;")
cur.fetchall()

[('Total Stock Market Index Fund', 6, 12, 0.5)]

In [7]:
cur.execute("""SELECT
        sec.sector_name,
        SUM(h.current_weight) as exposure
    FROM holdings h
    JOIN securities s
        ON h.security_id = s.security_id
    JOIN sectors sec
        ON s.sector_id = sec.sector_id
    WHERE
        h.portfolio_id = 7
        AND s.asset_type = 'Stock'
    GROUP BY sec.sector_name
    ORDER BY exposure DESC""")
cur.fetchall()

[('Energy', 0.38),
 ('Technology', 0.36),
 ('Automotive', 0.15),
 ('Consumer Staples', 0.11)]

Testing Exposure py

In [8]:
from src.tools.sql_query_tool import connect_to_db
from src.tools.exposure_tool import calculate_sector_exposure

db = connect_to_db()

result = calculate_sector_exposure(
    "International Equity Fund",
    db
)

print(result)

2026-04-05 17:51:31,103 | INFO | src.tools.sql_query_tool | Database path: D:\Pl_Capitals_Assignment\portfolio-analytics-agent\data\db\portfolio_database.db
2026-04-05 17:51:31,107 | INFO | src.tools.sql_query_tool | Database exists: True
2026-04-05 17:51:31,674 | INFO | src.tools.sql_query_tool | Database dialect: sqlite
2026-04-05 17:51:31,677 | INFO | src.tools.sql_query_tool | Available tables: ['benchmarks', 'historical_prices', 'holdings', 'portfolio_performance', 'portfolios', 'risk_metrics', 'sectors', 'securities', 'transactions']
2026-04-05 17:51:31,678 | INFO | src.tools.exposure_tool | Calculating sector exposure for identifier=International Equity Fund
2026-04-05 17:51:31,683 | INFO | src.tools.exposure_tool | Resolved portfolio 'International Equity Fund' to id 7
2026-04-05 17:51:31,686 | INFO | src.tools.exposure_tool | Exposure query executed successfully
2026-04-05 17:51:31,687 | INFO | src.tools.exposure_tool | Raw result: [('Energy', 0.38), ('Technology', 0.36), ('Au

{'portfolio_id': 7, 'sector_exposure': {'Energy': 0.38, 'Technology': 0.36, 'Automotive': 0.15, 'Consumer Staples': 0.11}}


In [9]:

# Agent Query & Response
cur.execute("SELECT p.portfolio_name, SUM(CASE WHEN sec.sector_name = 'Technology' THEN h.quantity * s.current_price ELSE 0 END) AS tech_value, SUM(h.quantity * s.current_price) AS total_value, (SUM(CASE WHEN sec.sector_name = 'Technology' THEN h.quantity * s.current_price ELSE 0 END) / SUM(h.quantity * s.current_price)) * 100 AS tech_percentage FROM holdings h JOIN portfolios p ON h.portfolio_id = p.portfolio_id JOIN securities s ON h.security_id = s.security_id JOIN sectors sec ON s.sector_id = sec.sector_id WHERE s.asset_type = 'Stock' GROUP BY p.portfolio_name ")
cur.fetchall()


[('Balanced Portfolio', 388704.0, 772304.0, 50.33043982680395),
 ('Conservative Income Fund', 0, 891920.0, 0.0),
 ('Dividend Aristocrats Fund', 0, 944180.0, 0.0),
 ('ESG Sustainable Fund', 272978.0, 1007228.0, 27.101907413217262),
 ('Emerging Markets Fund', 339095.0, 1045100.0, 32.44617739929193),
 ('Growth Equity Fund', 1866148.0, 2204148.0, 84.66527656037617),
 ('International Equity Fund', 346888.0, 734928.0, 47.20026995841769),
 ('Small Cap Value Fund', 0, 758085.0, 0.0),
 ('Tech Innovation Fund', 1565292.5, 1565292.5, 100.0),
 ('Total International Index Fund', 1427086.0, 2281686.0, 62.545240668523185),
 ('Total Stock Market Index Fund', 3093520.0, 6622340.0, 46.713397379174125)]

In [8]:

# Actual Ground Truth SQL Query and response
cur.execute("WITH portfolio_tech_value AS (SELECT p.portfolio_id, p.portfolio_name, SUM(h.quantity * s.current_price) as tech_value FROM portfolios p JOIN holdings h ON p.portfolio_id = h.portfolio_id JOIN securities s ON h.security_id = s.security_id JOIN sectors sec ON s.sector_id = sec.sector_id WHERE sec.sector_name = 'Technology' GROUP BY p.portfolio_id, p.portfolio_name), portfolio_total_value AS (SELECT p.portfolio_id, SUM(h.quantity * s.current_price) as total_value FROM portfolios p JOIN holdings h ON p.portfolio_id = h.portfolio_id JOIN securities s ON h.security_id = s.security_id GROUP BY p.portfolio_id) SELECT ptv.portfolio_name, ptv.tech_value, ptv2.total_value, ROUND((ptv.tech_value / ptv2.total_value) * 100, 2) as tech_percentage FROM portfolio_tech_value ptv JOIN portfolio_total_value ptv2 ON ptv.portfolio_id = ptv2.portfolio_id ORDER BY tech_percentage DESC;")
cur.fetchall()

[('Tech Innovation Fund', 1565292.5, 1565292.5, 100.0),
 ('Growth Equity Fund', 1866148.0, 2204148.0, 84.67),
 ('Total International Index Fund', 1427086.0, 2281686.0, 62.55),
 ('International Equity Fund', 346888.0, 734928.0, 47.2),
 ('Total Stock Market Index Fund', 3093520.0, 6622340.0, 46.71),
 ('Balanced Portfolio', 388704.0, 1073279.0, 36.22),
 ('Emerging Markets Fund', 339095.0, 1045100.0, 32.45),
 ('ESG Sustainable Fund', 272978.0, 1007228.0, 27.1)]